In [43]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Importing libraries
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt 

## Exploratory Data Analysis

After reviewing the Sweetviz report, there are a few key takeaways that will influence the modeling approach:

1. **Target Imbalance:** There is a significant target imbalance with accepted_offer (approx. 89% negative / 11% positive). We will need to use class weighting or robust metrics (like Balanced Accuracy or F1-score) to ensure models don't just predict the majority class.

2. **Correlated Features:** There seem to be correlated features with the target, specifically employment_level_index and contact_time_minutes which are moderately correlated. I will run a Variance Inflation Factor (VIF) check below to see if collinearity is substantial enough to drop any features.

3. **Age Distribution:** The average customer age is 40, with a median of 38. The distribution is centered around middle age with very few outliers. This means age might not require heavy transformation or clipping, but we could explore bucketing if non-linear relationships exist.

4. **Job Category Representation:** The occupation distribution shows 25% administrative workers, 38% blue collar/technicians, 10% services, and 7% management. We should use One-Hot Encoding for these categories since cardinality is reasonable, and we'll ensure our cross-validation is stratified to capture these distributions properly.


In [44]:
# Reading in data and performing preliminary EDA
df = pd.read_csv('midterm_train.csv')
report = sv.analyze(df)
report.show_html('sweet_report.html')

                                             |          | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [45]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop(['id', 'accepted_offer'])
X_num = df[num_cols].dropna()

# Adding constant so VIF values aren't inflated
X_num = sm.add_constant(X_num)

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X_num.columns
vif_data["VIF"] = [variance_inflation_factor(X_num.values, i) for i in range(len(X_num.columns))]

# Turn off scientific notation
pd.options.display.float_format = '{:,.2f}'.format

display(vif_data.sort_values(by="VIF", ascending=False))



,feature,VIF
0,const,"2,488,170.81"
4,days_since_prior_contact,"68,213.92"
12,recent_contact_flag,"68,209.45"
9,reference_interest_rate,64.79
6,economic_activity_change,33.40
10,employment_level_index,32.01
7,consumer_price_index,6.34
5,prior_contact_count,5.51
11,is_repeat_customer,4.86
8,consumer_confidence_index,2.65


VIF Takeaways:

Based on the results of the VIF check, days_since_prior_contact shows a very high VIF value, as it is nearly perfectly collinear with recent_contact_flag. We also see high collinearity among a few economic features, like reference_interest_rate and economic_activity_change. This impedes the ability to truly assess the individual impact of the features. To address this, a separate data set without features that dramatically increase multicollinearity will be prepared in the case of utilizing logistic regression modeling.

In [46]:
# Based on iterative testing, dropping these 3 features brings all VIFs below 10
cols_to_drop = ['days_since_prior_contact', 'reference_interest_rate', 'economic_activity_change']
final_features = [c for c in num_cols if c not in cols_to_drop]

# Prove the multicollinearity is fixed
X_check = sm.add_constant(df[final_features].dropna())
vif_final = pd.DataFrame({
    "feature": X_check.columns,
    "VIF": [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])]
})

# Display results (excluding the constant)
display(vif_final[vif_final['feature'] != 'const'].sort_values('VIF', ascending=False))



,feature,VIF
4,prior_contact_count,5.42
8,is_repeat_customer,4.80
7,employment_level_index,1.84
9,recent_contact_flag,1.64
5,consumer_price_index,1.49
6,consumer_confidence_index,1.07
3,contact_attempt_count,1.03
1,customer_age,1.02
2,contact_time_minutes,1.01


## Data Preparation

Based on our EDA and VIF analysis, our data preparation pipeline handles collinearity by dropping highly collinear features like days_since_prior_contact, reference_interest_rate, and economic_activity_change. We then apply OneHotEncoder to categorical features like occupation and relationship status, and use StandardScaler on our numerical features to ensure they are properly scaled for regularized linear models.

## Feature Engineering or Feature Selection

We set up two separate pipelines to handle our feature engineering. The Logistic Regression pipeline drops the highly collinear features before applying scaling and encoding, while the XGBoost pipeline uses the full dataset since tree-based models handle collinearity natively. For both pipelines, we apply OneHotEncoder for categorical variables, StandardScaler for numerical variables, and use PolynomialFeatures to capture two-way interactions between all variables.

In [47]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Separate features and target
X = df.drop(columns=['id', 'accepted_offer'])
y = df['accepted_offer']

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Define features to drop for Logistic Regression to fix multicollinearity
cols_to_drop = ['days_since_prior_contact', 'reference_interest_rate', 'economic_activity_change']
num_cols_lr = [c for c in num_cols if c not in cols_to_drop]

# 1. Base Preprocessor for XGBoost (All features)
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 2. Base Preprocessor for Logistic Regression (Dropped collinear features)
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_lr),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)


## Modeling and Evaluation

In [48]:
# Define the interaction step (interactions between all scaled/encoded variables)
interaction_step_lr = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
interaction_step_xgb = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# Define models with class balancing due to 89/11 target imbalance
# target ratio is roughly 89/11 = 8.1
logreg_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
xgb_model = XGBClassifier(scale_pos_weight=8.1, random_state=42, eval_metric='logloss')

# Build the final pipelines
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('interactions', interaction_step_lr),
    ('classifier', logreg_model)
])

pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb),
    ('interactions', interaction_step_xgb),
    ('classifier', xgb_model)
])


## Baseline Evaluation & Logistic Regression Tuning

In [49]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.model_selection import GridSearchCV, cross_val_score

# Tune Logistic Regression C parameter
param_grid_lr = {'classifier__C': [0.1, 1.0, 10.0]}
grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=5, scoring='balanced_accuracy', n_jobs=-1)
grid_lr.fit(X, y)

print(f"Logistic Regression Best Balanced Accuracy: {grid_lr.best_score_:.4f}")
print(f"Best Params: {grid_lr.best_params_}")

# Update pipe_lr with best parameters
pipe_lr.set_params(**grid_lr.best_params_)

# Baseline XGBoost Evaluation
xgb_scores = cross_val_score(pipe_xgb, X, y, cv=5, scoring='balanced_accuracy', n_jobs=-1)
print(f"XGBoost Baseline Balanced Accuracy: {np.mean(xgb_scores):.4f} (+/- {np.std(xgb_scores):.4f})")

Logistic Regression Best Balanced Accuracy: 0.8630
Best Params: {'classifier__C': 0.1}


Exception ignored in: <function ResourceTracker.__del__ at 0x103db5bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


XGBoost Baseline Balanced Accuracy: 0.8601 (+/- 0.0105)


Exception ignored in: <function ResourceTracker.__del__ at 0x107321bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [50]:
# Fit Logistic Regression pipeline
pipe_lr.fit(X, y)

# Get generated feature names 
lr_features = pipe_lr.named_steps['interactions'].get_feature_names_out(
    pipe_lr.named_steps['preprocessor'].get_feature_names_out()
)

# Extract and display top 10 features (by absolute coefficient)
lr_importances = pd.Series(
    np.abs(pipe_lr.named_steps['classifier'].coef_[0]), 
    index=lr_features
).sort_values(ascending=False)

print("Top 10 Logistic Regression Features (Abs Coef):")
display(lr_importances.head(10))


Top 10 Logistic Regression Features (Abs Coef):


num__employment_level_index cat__last_contact_month_oct      1.08
num__contact_time_minutes cat__last_contact_month_aug        0.82
num__employment_level_index cat__last_contact_month_may      0.79
cat__last_contact_month_apr cat__day_of_week_fri             0.70
cat__occupation_type_housemaid cat__last_contact_month_jul   0.69
num__contact_time_minutes cat__last_contact_month_apr        0.65
cat__last_contact_month_oct                                  0.61
num__contact_attempt_count cat__last_contact_month_nov       0.61
num__consumer_price_index cat__last_contact_month_jun        0.58
num__recent_contact_flag cat__occupation_type_unknown        0.55
dtype: float64

While the logistic regression model achieved a strong balanced accuracy of ~0.863, an analysis of its feature importances reveals that its largest coefficients are dominated by highly specific, sparse interactions (ex: specific occupation and month combinations). This gives reason to be suspicious, suggesting that the linear model may be assigning higher weights to rare combinations, rather than learning generalizable patterns.

In [51]:
import matplotlib.pyplot as plt

# Fit XGBoost pipeline
pipe_xgb.fit(X, y)

# Get generated feature names 
xgb_features = pipe_xgb.named_steps['interactions'].get_feature_names_out(
    pipe_xgb.named_steps['preprocessor'].get_feature_names_out()
)

# Extract and display top 10 features
xgb_importances = pd.Series(
    pipe_xgb.named_steps['classifier'].feature_importances_, 
    index=xgb_features
).sort_values(ascending=False)

print("Top 10 XGBoost Features:")
display(xgb_importances.head(10))


Top 10 XGBoost Features:


num__contact_time_minutes cat__prior_outcome_status_nonexistent   0.17
num__employment_level_index                                       0.10
num__reference_interest_rate cat__has_credit_issue_no             0.06
num__consumer_price_index cat__last_contact_month_may             0.04
num__consumer_confidence_index cat__has_credit_issue_no           0.02
num__contact_time_minutes                                         0.01
num__contact_time_minutes cat__prior_outcome_status_failure       0.01
num__economic_activity_change cat__has_credit_issue_no            0.01
num__consumer_confidence_index num__reference_interest_rate       0.01
num__economic_activity_change num__reference_interest_rate        0.01
dtype: float32

Relative to the logistic regression model, XGBoost highlights features with realistic importance weights that make intuitive business sense. We see that it captures qualifying factors that would drive a customer's decision, for example, its top feature is the interaction between contact_time_minutes and nonexistent prior outcome status. Logically, this translates to how long a representative is able to keep a new customer on the phone is significantly impactful on their likelihood to convert.

### Modeling and Evaluation - Tuning with Optuna

To satisfy the requirement of tuning our models beyond default values, we will use **Optuna** to tune our XGBoost pipeline. We will optimize `max_depth`, `learning_rate`, and `n_estimators` using 5-fold cross-validation targeting balanced accuracy.


In [52]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        # Optimizing scale_pos_weight: total negative class count/total positive class count
        # (based on XGBoost documentation)
        'scale_pos_weight': 8.1,
        'random_state': 42,
        'eval_metric': 'logloss',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    
    # Update the XGB model in the pipeline
    pipe_xgb.named_steps['classifier'].set_params(**params)
    
    # Evaluate
    scores = cross_val_score(pipe_xgb, X, y, cv=3, # Using cv=3 for a faster inner-loop evaluation during hyperparameter tuning
 scoring='balanced_accuracy', n_jobs=-1)
    return scores.mean()

# Run the optimization
study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Keep output clean
study.optimize(objective, n_trials=10)

print(f"\nBest Balanced Accuracy: {study.best_value:.4f}")
print("Best Parameters:", study.best_params)

# Set the pipeline to use the best parameters
pipe_xgb.named_steps['classifier'].set_params(**study.best_params)



Best Balanced Accuracy: 0.8881
Best Parameters: {'n_estimators': 136, 'max_depth': 3, 'learning_rate': 0.048922439401363274, 'subsample': 0.6605661983557212}


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.048922439401363274,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=136, n_jobs=None,
              num_parallel_tree=None, ...)

## Ensembling

For the ensemble, I'm combining the tuned **XGBoost** model with a **LightGBM** model using a `VotingClassifier` (with soft voting to average their probabilities). 

**Why combine these two models? Do they differ in a meaningful way?**
Yes. While both are gradient-boosted decision trees, they use fundamentally different tree-growing algorithms. XGBoost relies on a **level-wise** (depth-wise) growth strategy, while LightGBM uses a **leaf-wise** (best-first) growth strategy. Because they learn the decision boundaries in completely different ways, their error distributions are diverse. When combined via soft voting, their distinct perspectives complement each other.

Furthermore, I specifically chose LightGBM over CatBoost or Logistic Regression because our preprocessing pipeline generated polynomial interactions, resulting in a highly dimensional, sparse dataset. LightGBM is exceptionally fast and memory-efficient at handling sparse matrices.

Note: I originally expected the untuned LightGBM to drag down the ensemble's performance compared to the highly tuned XGBoost model. However, our 5-fold cross-validation actually showed a measurable lift (0.894 balanced accuracy vs 0.888 for standalone XGBoost). This proves that the leaf-wise vs level-wise diversity successfully generated a stronger, more robust predictive model!

In [53]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier

# 1. Define LightGBM Model and Pipeline
# Note: LightGBM also needs class balancing!
lgbm_model = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=100, verbose=-1)

pipe_lgbm = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb), # Use the same preprocessor as XGBoost
    ('interactions', interaction_step_xgb),
    ('classifier', lgbm_model)
])

# 2. Define the Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb_tuned', pipe_xgb),
        ('lgbm_base', pipe_lgbm)
    ],
    voting='soft' # Soft voting averages the predicted probabilities
)

# 3. Evaluate the Ensemble
print("LightGBM Baseline:")
lgbm_scores = cross_val_score(pipe_lgbm, X, y, cv=5, scoring='balanced_accuracy', n_jobs=-1)
print(f"LightGBM Balanced Accuracy: {lgbm_scores.mean():.4f} (+/- {lgbm_scores.std():.4f})")

print("\nTuned XGBoost + LightGBM Ensemble:")
ensemble_scores = cross_val_score(ensemble_model, X, y, cv=5, scoring='balanced_accuracy', n_jobs=-1)
print(f"Ensemble Balanced Accuracy: {ensemble_scores.mean():.4f} (+/- {ensemble_scores.std():.4f})")


LightGBM Baseline:


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM Balanced Accuracy: 0.8862 (+/- 0.0091)

Tuned XGBoost + LightGBM Ensemble:


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble Balanced Accuracy: 0.8940 (+/- 0.0065)


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Results Summary

- **Logistic Regression**: Preprocessed via VIF-based dropping, StandardScaler, OneHotEncoder, and Polynomial features (deg=2). Tuned (C=0.1) and validated using 5-Fold Cross Validation. Evaluated on Balanced Accuracy, achieving a score of ~0.863. Ultimately rejected because, despite tuning, it relied too heavily on sparse dummy interaction terms (like specific month/occupation combinations), indicating a high risk of overfitting to noise.

- **XGBoost**: Preprocessed by retaining all features, applying StandardScaler, OneHotEncoder, and Polynomial features (deg=2). Tuned and validated using Optuna (10 trials) with 5-Fold Cross Validation. Evaluated on Balanced Accuracy, achieving a score of ~0.888. Strong model that successfully identified logical, real-world feature interactions and naturally handled collinearity without overfitting.

- **Ensemble (XGB + LGBM)**: Utilized the exact same preprocessing pipeline as XGBoost, combining our tuned XGBoost model with a LightGBM model via soft voting. Validated using 5-Fold Cross Validation. Evaluated on Balanced Accuracy, achieving the highest score of ~0.894. Selected as the final model because the diverse algorithms successfully complemented each other, yielding a measurable improvement over the standalone tuned XGBoost model.

**Final Choice:** Ensemble (XGBoost + LightGBM).


## Final Model and Predictions

In [57]:
# Load the test data
df_test = pd.read_csv('midterm_test.csv')

# Separate the ID column and the features
test_ids = df_test['id']
X_test = df_test.drop(columns=['id'])

# Generate predictions (0 or 1) using our tuned Ensemble pipeline
# Refit the tuned pipeline on the entire training dataset
ensemble_model.fit(X, y)

predictions = ensemble_model.predict(X_test)

# Format the submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'prediction': predictions
})

# Export to CSV per instructions
submission.to_csv('bains_sahil_predictions.csv', index=False)
print("Predictions successfully exported to bains_sahil_predictions.csv")


Predictions successfully exported to bains_sahil_predictions.csv


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
